In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import imageio
from xarray.coders import CFDatetimeCoder

# =========================================================
# Plot style (Nature Geoscience vibe)
# =========================================================

def set_plot_style():
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 8,
        "axes.titlesize": 8,
        "axes.labelsize": 8,
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "xtick.major.width": 0.8,
        "ytick.major.size": 3,
        "ytick.major.width": 0.8,
        "savefig.dpi": 300,
        "figure.dpi": 300,
        "figure.facecolor": "white",
        "text.color": "black",
        "axes.edgecolor": "black",
        "xtick.color": "black",
        "ytick.color": "black",
    })


# =========================================================
# Utilities
# =========================================================

def compute_edges(coord):
    """
    Given 1D coord, return cell-edge coordinates for pcolormesh.
    Works for numpy arrays or 1D xarray DataArrays.
    """
    coord = np.asarray(coord)
    d = np.diff(coord) / 2.0
    return np.concatenate([[coord[0] - d[0]], coord[:-1] + d, [coord[-1] + d[-1]]])


def plev_to_hPa(plev_da):
    """
    Convert model plev coord to hPa.
    Assumes:
      - If max(p) > 2000, p is in Pa -> divide by 100.
      - Else assume already hPa.
    Returns xr.DataArray (plev).
    """
    p = plev_da.values.astype(float)
    if np.nanmax(p) > 2000:
        p_hPa = p / 100.0  # Pa -> hPa
    else:
        p_hPa = p
    return xr.DataArray(p_hPa, coords={'plev': plev_da}, dims=['plev'])


def pressure_to_height(plev_da, p0=1e5, H=7000.0):
    """
    Convert pressure to ~log-pressure height (km).
    p0 in Pa, H [m] is scale height.
    If plev is in hPa (<2000), convert to Pa first.
    Returns xr.DataArray height_km(plev).
    """
    p = plev_da.values.astype(float)
    if np.nanmax(p) < 2000:  # likely already in hPa
        p = p * 100.0        # to Pa
    z_km = -H * np.log(p / p0) / 1000.0
    return xr.DataArray(z_km, coords={'plev': plev_da}, dims=['plev'])


def load_hindcast(pattern, start, end):
    """
    Load hindcast experiment files (short hindcasts),
    slice time range [start, end],
    then average across 'member' dimension.
    """
    coder = CFDatetimeCoder(use_cftime=True)
    files = sorted(glob.glob(pattern))
    dss = []
    for f in files:
        ds = xr.open_dataset(f, decode_times=coder)
        sel = ds.sel(time=slice(start, end))
        if sel.time.size > 0:
            dss.append(sel)

    if not dss:
        raise RuntimeError(f"No hindcast data for pattern {pattern}")

    ds_all = xr.concat(dss, dim='member')
    ds_all = ds_all.mean(dim='member')
    return ds_all


def load_control(path, start, end):
    """
    Load long 'Reference' integration for same time window.
    """
    coder = CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(path, decode_times=coder)
    sel = ds.sel(time=slice(start, end))
    if sel.time.size == 0:
        raise RuntimeError(f"No control data for slice {start}–{end}")
    return sel


# =========================================================
# O3 DU/km conversion
# =========================================================

def compute_o3_dukm_full(ds):
    """
    Convert O3 mixing ratio (vmr, mol/mol) to DU/km using co-located T, plev.
    Formula:
        DU/km = 26.956 * [ P(hPa) / T(K) ] * O3_ppmv
    where O3_ppmv = O3 * 1e6  (if O3 is vmr mol/mol).

    Inputs:
        ds must have: O3(time, plev, lat, lon),
                      T(time,  plev, lat, lon),
                      plev(plev)

    Returns:
        o3_dukm(time, plev, lat, lon) in DU/km
    """
    if ('O3' not in ds) or ('T' not in ds) or ('plev' not in ds):
        raise KeyError("Dataset missing O3, T, or plev for DU/km calc.")

    O3 = ds['O3']  # vmr mol/mol
    T  = ds['T']   # K
    p_hPa = plev_to_hPa(ds['plev'])  # (plev)

    # broadcast p_hPa across (time,plev,lat,lon)
    O3_ppmv = O3 * 1e6
    COEFF = 26.95560296256018  # derived constant
    o3_dukm = COEFF * p_hPa * O3_ppmv / T

    o3_dukm.attrs['units'] = 'DU/km'
    o3_dukm.attrs['long_name'] = 'O3 local column density per km'
    return o3_dukm


# =========================================================
# Tropopause helpers (kept for future use, but NOT plotted now)
# =========================================================

def _find_lapse_rate_tropopause_single_col(T_prof, z_prof, min_layer_km=2.0):
    """
    Returns ~tropopause height (km) using a lapse-rate criterion with
    cold-point fallback. NOT USED IN CURRENT PLOTS.
    """
    idx = np.argsort(z_prof)
    z_sorted = z_prof[idx]
    T_sorted = T_prof[idx]

    mask = np.isfinite(z_sorted) & np.isfinite(T_sorted)
    if np.sum(mask) < 3:
        i_min = np.nanargmin(T_prof)
        return z_prof[i_min]

    z_sorted = z_sorted[mask]
    T_sorted = T_sorted[mask]

    dz = np.diff(z_sorted)
    dT = np.diff(T_sorted)
    with np.errstate(divide='ignore', invalid='ignore'):
        lapse = -(dT / dz)  # K/km

    for i in range(len(lapse)):
        if not np.isfinite(lapse[i]):
            continue
        if lapse[i] <= 2.0:
            z0 = z_sorted[i]
            z_top = z0 + min_layer_km

            upper_idx = np.where(z_sorted[:-1] >= z0)[0]
            upper_idx = upper_idx[z_sorted[upper_idx] <= z_top]
            if upper_idx.size == 0:
                mean_lapse = lapse[i]
            else:
                layer_inds = [k for k in upper_idx if k < len(lapse)]
                if not layer_inds:
                    mean_lapse = lapse[i]
                else:
                    mean_lapse = np.nanmean(lapse[layer_inds])

            if np.isfinite(mean_lapse) and mean_lapse <= 2.0:
                return z0

    i_min = np.nanargmin(T_prof)
    return z_prof[i_min]


def compute_tropopause_heights(T2d, height_km, min_layer_km=2.0):
    """
    Lapse-rate tropopause vs latitude, with cold-point fallback.
    NOT USED IN CURRENT PLOTS.
    """
    T_vals = T2d.values        # (plev, lat)
    z_vals = height_km.values  # (plev,)
    nlat   = T_vals.shape[1]

    tropo = np.full(nlat, np.nan)
    for j in range(nlat):
        T_prof = T_vals[:, j]
        tropo[j] = _find_lapse_rate_tropopause_single_col(
            T_prof, z_vals, min_layer_km=min_layer_km
        )
    return tropo


# =========================================================
# Plotting / GIF routines
# =========================================================

def plot_spliced_seasonal_average(datasets, lat, height_km, output_file):
    """
    Seasonal-mean style static panel plot (2x3 grid):
    - Color: O3 in DU/km (time+lon mean)
    - Contours: U (20/30/40/50 m/s), time+lon mean
    - NO tropopause line (removed per request)
    """
    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    tags = list(datasets.keys())

    fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                             sharex=True, sharey=True)
    axes = axes.flatten()

    pcm = None
    for i, tag in enumerate(tags):
        ds = datasets[tag]

        # O3 DU/km, then mean over time+lon
        o3_dukm = compute_o3_dukm_full(ds)             # (time,plev,lat,lon)
        o3_mean = o3_dukm.mean(dim=['time', 'lon'])     # (plev,lat)

        # wind climatology:
        u_mean  = ds['U'].mean(dim=['time', 'lon'])     # (plev,lat)

        ax = axes[i]
        pcm = ax.pcolormesh(
            LAT_e, HT_e, o3_mean.values,
            shading='auto', cmap='rainbow'
        )

        ct  = ax.contour(
            LATc, HTc, u_mean.values,
            levels=[20, 30, 40, 50],
            colors='black', linewidths=1
        )
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

        ax.set_title(f'({chr(97+i)}) {tag}')
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)

        # thin spines
        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

    # remove any unused last panel(s) if <6 tags
    if len(tags) < len(axes):
        for k in range(len(tags), len(axes)):
            fig.delaxes(axes[k])

    # shared colorbar
    cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
    cb.set_label('O₃ [DU/km]')

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    fig.savefig(output_file, dpi=300)
    plt.close(fig)


def create_o3_gif(ds, lat, height_km, frames_dir, gif_file, duration=0.5):
    """
    Single-scenario animated cross-section over time:
    - Color: O3 DU/km (lon mean)
    - Contours: U (20/30/40/50 m/s, lon mean)
    - NO tropopause line
    Uses a fixed color scale across frames for that scenario.
    """
    os.makedirs(frames_dir, exist_ok=True)

    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    # Precompute O3 DU/km (time,plev,lat,lon)
    o3_dukm_full = compute_o3_dukm_full(ds)
    # Lon-mean, keep time dimension
    o3_dukm_lonmean = o3_dukm_full.mean(dim='lon')  # (time,plev,lat)

    # global color scale for this dataset
    vmin = float(np.nanmin(o3_dukm_lonmean.values))
    vmax = float(np.nanmax(o3_dukm_lonmean.values))

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)

    for i, t in enumerate(ds.time.values):
        o3_now = o3_dukm_lonmean.isel(time=i)           # (plev,lat)
        u_now  = ds['U'].isel(time=i).mean(dim='lon')   # (plev,lat)

        fig, ax = plt.subplots(figsize=(8, 6))
        pcm = ax.pcolormesh(
            LAT_e, HT_e, o3_now.values,
            shading='auto', cmap='rainbow',
            vmin=vmin, vmax=vmax
        )

        ct = ax.contour(
            LATc, HTc, u_now.values,
            levels=[20, 30, 40, 50],
            colors='balck', linewidths=1
        )
        ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

        ax.set_title(str(t))
        ax.set_xlabel('Latitude (°)')
        ax.set_ylabel('Height (km)')
        ax.set_ylim(0, 30)

        for spine in ax.spines.values():
            spine.set_linewidth(0.8)

        # add colorbar for each frame so units stay visible in GIF
        cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('O₃ [DU/km]')

        frame = os.path.join(frames_dir, f'frame_{i:03d}.png')
        fig.savefig(frame, dpi=200)
        plt.close(fig)

        writer.append_data(imageio.imread(frame))

    writer.close()


def create_combined_gif(datasets, lat, height_km,
                        frames_dir, combo_file,
                        duration=0.5):
    """
    Multi-panel animated comparison:
    - Each panel is one dataset
    - Color: O3 DU/km (lon mean)
    - Contours: U (lon mean)
    - NO tropopause line
    - Shared colorbar per frame (consistent vmin/vmax across all datasets+all times)
    """
    os.makedirs(frames_dir, exist_ok=True)

    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    tags = list(datasets.keys())

    # Precompute DU/km lon-mean for every dataset
    dukm_lonmean = {}
    for tag, ds in datasets.items():
        o3_dukm_full = compute_o3_dukm_full(ds)           # (time,plev,lat,lon)
        dukm_lonmean[tag] = o3_dukm_full.mean(dim='lon')   # (time,plev,lat)

    # Time axis (assume same for all)
    times = next(iter(datasets.values())).time.values

    # Global color range across ALL datasets and ALL frames
    all_vals = []
    for tag in tags:
        all_vals.append(dukm_lonmean[tag].values)
    all_vals = np.concatenate([a.reshape(-1) for a in all_vals])
    vmin = float(np.nanmin(all_vals))
    vmax = float(np.nanmax(all_vals))

    writer = imageio.get_writer(combo_file, mode='I', duration=duration)

    for ti, t in enumerate(times):
        fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                                 sharex=True, sharey=True)
        axes = axes.flatten()

        pcm = None
        for j, tag in enumerate(tags):
            ds   = datasets[tag]
            o3_2d = dukm_lonmean[tag].isel(time=ti)        # (plev,lat)
            u2d   = ds['U'].isel(time=ti).mean(dim='lon')  # (plev,lat)

            ax = axes[j]
            pcm = ax.pcolormesh(
                LAT_e, HT_e, o3_2d.values,
                shading='auto', cmap='rainbow',
                vmin=vmin, vmax=vmax
            )
            ct  = ax.contour(
                LATc, HTc, u2d.values,
                levels=[20, 30, 40, 50],
                colors='balck', linewidths=1
            )
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

            ax.set_title(f'({chr(97+j)}) {tag}')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)

            for spine in ax.spines.values():
                spine.set_linewidth(0.8)

        # remove unused last axes if <6 tags
        if len(tags) < len(axes):
            for k in range(len(tags), len(axes)):
                fig.delaxes(axes[k])

        # shared colorbar
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('O₃ [DU/km]')

        fig.suptitle(str(t))
        plt.tight_layout(rect=[0, 0, 0.9, 0.95])

        frame = os.path.join(frames_dir, f'combo_{ti:03d}.png')
        fig.savefig(frame, dpi=250)
        plt.close(fig)

        writer.append_data(imageio.imread(frame))

    writer.close()


def create_spliced_difference_gif(datasets, lat, height_km,
                                  frames_dir, gif_file,
                                  duration=0.5):
    """
    Animated difference vs 'Reference' (2×2 panels for 4 hindcast scenarios):
    - Color: ΔO3 DU/km = O3_exp(DU/km) − O3_ref(DU/km), lon-mean
    - Contours: U of experiment (lon mean)
    - NO tropopause line
    - Color scale: symmetric across ALL experiments/time
    """
    os.makedirs(frames_dir, exist_ok=True)

    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    # Which datasets are non-reference experiments?
    exp_tags = [t for t in datasets if t != 'Reference']

    # Precompute DU/km lon-mean for each dataset
    dukm_lonmean = {}
    for tag, ds in datasets.items():
        o3_dukm_full = compute_o3_dukm_full(ds)           # (time,plev,lat,lon)
        dukm_lonmean[tag] = o3_dukm_full.mean(dim='lon')   # (time,plev,lat)

    # Reference DU/km
    ref_dukm = dukm_lonmean['Reference']  # (time,plev,lat)

    # Assume all experiments share same time stamps/length:
    times = datasets[exp_tags[0]].time.values

    # Build global symmetric color range for diffs:
    diffs_flat = []
    for tag in exp_tags:
        diff_all = dukm_lonmean[tag] - ref_dukm  # (time,plev,lat)
        diffs_flat.append(diff_all.values.reshape(-1))
    diffs_flat = np.concatenate(diffs_flat)

    max_abs = float(np.nanmax(np.abs(diffs_flat)))
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0
    vmin, vmax = -max_abs, max_abs

    writer = imageio.get_writer(gif_file, mode='I', duration=duration)

    for ti, t in enumerate(times):
        fig, axes = plt.subplots(2, 2, figsize=(12, 8),
                                 sharex=True, sharey=True)
        axes = axes.flatten()

        pcm = None
        for j, tag in enumerate(exp_tags):
            ds       = datasets[tag]
            diff2d   = (dukm_lonmean[tag].isel(time=ti)
                        - ref_dukm.isel(time=ti))         # (plev,lat)
            u2d      = ds['U'].isel(time=ti).mean(dim='lon')

            ax = axes[j]
            pcm = ax.pcolormesh(
                LAT_e, HT_e, diff2d.values,
                shading='auto', cmap='RdBu_r',
                vmin=vmin, vmax=vmax
            )
            ct  = ax.contour(
                LATc, HTc, u2d.values,
                levels=[20, 30, 40, 50],
                colors='black', linewidths=1
            )
            ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

            ax.set_title(f'({chr(97+j)}) {tag} – Reference')
            ax.set_xlabel('Latitude (°)')
            ax.set_ylabel('Height (km)')
            ax.set_ylim(0, 30)

            for spine in ax.spines.values():
                spine.set_linewidth(0.8)

        # shared colorbar
        cax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
        cb = fig.colorbar(pcm, cax=cax, orientation='vertical')
        cb.set_label('ΔO₃ [DU/km]')

        fig.suptitle(str(t))
        plt.tight_layout(rect=[0, 0, 0.9, 0.95])

        frame = os.path.join(frames_dir, f'diff_{ti:03d}.png')
        fig.savefig(frame, dpi=250)
        plt.close(fig)

        writer.append_data(imageio.imread(frame))

    writer.close()


def create_snapshot_panel(datasets, lat, height_km, tag, out_png):
    """
    Static single snapshot for one scenario (first timestep):
    - Color: O3 DU/km (lon mean)
    - Contours: U (20/30/40/50 m/s, lon mean)
    - NO tropopause line
    """
    ds0 = datasets[tag].isel(time=0)

    # DU/km lon-mean at this snapshot
    o3_dukm_full = compute_o3_dukm_full(datasets[tag])  # (time,plev,lat,lon)
    o3_now = o3_dukm_full.isel(time=0).mean(dim='lon')  # (plev,lat)

    U_lon  = ds0['U'].mean(dim='lon')                   # (plev,lat)

    # mesh for pcolormesh / contour
    lat_e = compute_edges(lat)
    h_e   = compute_edges(height_km.values)
    LAT_e, HT_e = np.meshgrid(lat_e, h_e)
    LATc, HTc   = np.meshgrid(lat, height_km.values)

    fig, ax = plt.subplots(figsize=(10, 6))

    pcm = ax.pcolormesh(
        LAT_e, HT_e, o3_now.values,
        shading='auto', cmap='rainbow'
    )

    ct = ax.contour(
        LATc, HTc, U_lon.values,
        levels=[20, 30, 40, 50],
        colors='black', linewidths=1
    )
    ax.clabel(ct, fmt='%d', inline=True, fontsize=8)

    ax.set_xlabel('Latitude (°)')
    ax.set_ylabel('Height (km)')
    ax.set_ylim(0, 30)

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    cbar = fig.colorbar(pcm, ax=ax, orientation='vertical', pad=0.02)
    cbar.set_label('O₃ [DU/km]')

    fig.tight_layout()
    fig.savefig(out_png, dpi=300)
    plt.close(fig)


# =========================================================
# Main workflow
# =========================================================

if __name__ == "__main__":
    # --- choose period ---
    year  = '0008'
    start, end = "0008-03-01", "0008-05-31"

    # --- output dir ---
    PARENT = "/home/weiji/restart_exam/plots/O3_DUkm"
    os.makedirs(PARENT, exist_ok=True)

    # --- apply publication-like styling ---
    set_plot_style()

    # --- data patterns ---
    patterns = [
        (f"{year}_Feb_couple",
         f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
         f"BWCN.e122.f19_g16.002_{year}/Feb/"
         f"BWCN.e122.f19_g16.002.{year}-02.*.nc"),
        (f"{year}_Mar_couple",
         f"/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
         f"BWCN.e122.f19_g16.002_{year}/Mar/"
         f"BWCN.e122.f19_g16.002.{year}-03.*.nc"),
        (f"{year}_Feb_nocouple",
         f"/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/"
         f"{year}-02/*.nc"),
        (f"{year}_Mar_nocouple",
         f"/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/"
         f"{year}-03/*.nc"),
    ]

    control_file = (
        "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
        "BWCN.e122.f19_g16.002/"
        "BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
    )

    # --- load experiments + reference ---
    datasets = { tag: load_hindcast(pat, start, end) for tag, pat in patterns }
    datasets['Reference'] = load_control(control_file, start, end)

    # --- common coords ---
    example   = next(iter(datasets.values()))
    lat       = example.lat.values        # 1D latitude
    plev      = example.plev              # pressure levels
    height_km = pressure_to_height(plev)  # DataArray [plev] in km

    # 1) Seasonal mean style multi-panel static figure
    plot_spliced_seasonal_average(
        datasets, lat, height_km,
        os.path.join(PARENT, "MAM_average_spliced.png")
    )

    # 2) Individual scenario GIFs
    for tag, ds in datasets.items():
        scen_dir = os.path.join(PARENT, tag)
        frames   = os.path.join(scen_dir, "frames")
        gif_file = os.path.join(scen_dir, f"O3_{tag}.gif")
        os.makedirs(scen_dir, exist_ok=True)
        create_o3_gif(ds, lat, height_km, frames, gif_file)

    # 3) Combined GIF (all datasets in subplots)
    combo_frames = os.path.join(PARENT, "combined_frames")
    combo_gif    = os.path.join(PARENT, "O3_combined.gif")
    create_combined_gif(
        datasets, lat, height_km,
        combo_frames, combo_gif,
        duration=0.5
    )

    # 4) Difference GIF (experiment - Reference) in DU/km
    diff_frames = os.path.join(PARENT, "diff_frames")
    diff_gif    = os.path.join(PARENT, "O3_diff_combined.gif")
    create_spliced_difference_gif(
        datasets, lat, height_km,
        diff_frames, diff_gif,
        duration=0.5
    )

    # 5) One snapshot panel (first timestep of Feb_couple),
    #    using DU/km, NO tropopause line
    tag0 = f"{year}_Feb_couple"
    snapshot_png = os.path.join(PARENT, f"{tag0}_snapshot_t0.png")
    create_snapshot_panel(datasets, lat, height_km, tag0, snapshot_png)
